# 🧬 08 · 宏基因组基础：OTU 与 α 多样性

> 宏基因组直接从环境样本提取全部 DNA：肠道菌群、土壤微生物、
> 海洋微生物。第一站通常是 **16S rRNA 扩增子**测序：
> 序列 → OTU/ASV → 丰度表 → 多样性分析。本节手写：OTU 聚类、
> 丰度矩阵、**α 多样性（Shannon/Simpson/Chao1）**、稀疏曲线。

## 📋 本节要解决的问题

- OTU vs ASV 的区别？（97% 相似聚类 vs 单碱基精确）
- α 多样性度量什么？Shannon/Simpson/Chao1 各怎么算？
- 稀疏曲线（rarefaction curve）读什么？
- 为什么测序深度不同不能直接比较多样性指数？

## 1️⃣ OTU / ASV 概念

**OTU**（操作分类单元）：16S 序列按 97% 相似度聚成的类，近似"物种"。
**ASV**（扩增子序列变体）：不聚类，按单碱基差异精确定义，可跨研究比较。

流程：质控 → 去噪（DADA2 等）→ **特征表**（行=OTU/ASV，列=样本）→ 多样性。

In [1]:
# 简化 OTU 聚类：按 97% 相似度（此处用固定阈值示意）
def simple_otu_cluster(seqs, threshold=0.97):
    "最简贪心聚类：序列与已有代表序列相似度>=threshold 则归入，否则新OTU"
    def sim(s1, s2):
        same = sum(1 for a, b in zip(s1, s2) if a == b)
        return same / max(len(s1), len(s2), 1)
    reps = []
    otus = []
    for s in seqs:
        for i, r in enumerate(reps):
            if sim(s, r) >= threshold:
                otus[i].append(s)
                break
        else:
            reps.append(s)
            otus.append([s])
    return reps, otus

seqs = [
    'A'*100, 'A'*99 + 'C', 'A'*98 + 'CG',   # 聚类1：>97% 相似（1-2 个错配）
    'G'*100, 'G'*98 + 'CT',                  # 聚类2：>97% 相似（2 个错配）
    'C'*100,                                 # 聚类3：与上面差异大
]
reps, otus = simple_otu_cluster(seqs, 0.97)
print('OTU 数量(期望 3):', len(otus))
for i, o in enumerate(otus):
    print('  OTU%d: %d 条序列, 代表序列 %s...' % (i + 1, len(o), reps[i][:12]))
assert len(otus) == 3
print('✅ 简化 OTU 聚类完成（97% 相似度分组）')

OTU 数量(期望 3): 3
  OTU1: 3 条序列, 代表序列 AAAAAAAAAAAA...
  OTU2: 2 条序列, 代表序列 GGGGGGGGGGGG...
  OTU3: 1 条序列, 代表序列 CCCCCCCCCCCC...
✅ 简化 OTU 聚类完成（97% 相似度分组）


## 2️⃣ 手算示例：Shannon 与 Simpson 指数

> 一个样本读到 50 条序列：**物种 A=25, B=15, C=10**（比例 0.5/0.3/0.2）

**Shannon**：$H = -\sum_i p_i \ln p_i$
$$= -(0.5\ln0.5 + 0.3\ln0.3 + 0.2\ln0.2) = -(-0.347-0.361-0.322) \approx 1.030$$

**Simpson**：$D = \sum_i p_i^2 = 0.25 + 0.09 + 0.04 = 0.38$
常用 **1−D = 0.62**（越大多样性越高）。

> 对比：若 50 条序列全属一种，H=0、1−D=0（最不丰富）；
> 各占 1/50 种 → 接近最大值 ln(50)≈3.9、1−D≈0.98（极丰富）。

In [2]:
# 验证手算
import numpy as np
counts = np.array([25, 15, 10])
p = counts / counts.sum()
H = -(p * np.log(p)).sum()
D = (p**2).sum()
print('比例 =', p)
print('Shannon H(手算≈1.0297) =', round(H, 4))
print('Simpson D(手算=0.38) =', round(D, 4), ' 1-D =', round(1-D, 4))
assert abs(H - 1.0297) < 1e-3 and abs(D - 0.38) < 1e-9
print('✅ α 多样性手算验证通过')

比例 = [0.5 0.3 0.2]
Shannon H(手算≈1.0297) = 1.0297
Simpson D(手算=0.38) = 0.38  1-D = 0.62
✅ α 多样性手算验证通过


## 3️⃣ Chao1 物种丰富度估计

基于**稀有物种**（singleton=出现 1 次、doubleton=2 次）估计总物种数：
$$S_{Chao1} = S_{obs} + \frac{F_1^2}{2F_2}$$
其中 $S_{obs}$ 观察物种数，$F_1$ 单例物种数，$F_2$ 双例物种数。
> 若 F₁ 大 → 可能有大量未被采到的稀有物种 → 估计值远大于观察值。

In [3]:
def chao1(species_counts):
    "species_counts: 每个物种的观测计数列表"
    S_obs = len(species_counts)
    F1 = sum(1 for c in species_counts if c == 1)
    F2 = sum(1 for c in species_counts if c == 2)
    if F2 == 0:
        est = S_obs + (F1 * (F1 - 1)) / 2   # 无 doubleton 时的估计
    else:
        est = S_obs + F1**2 / (2 * F2)
    return est, S_obs, F1, F2

# 手算示例数据: 物种计数 [10, 8, 3, 1, 1] -> S=5, F1=2, F2=0
est, S, f1, f2 = chao1([10, 8, 3, 1, 1])
print('数据 [10,8,3,1,1]: S_obs=%d F1=%d F2=%d' % (S, f1, f2))
print('Chao1 估计(手算 5+2*1/2=6) =', est)
assert est == 6 and S == 5
print('✅ Chao1 手算验证通过（F1=2 → 估计 6）')

数据 [10,8,3,1,1]: S_obs=5 F1=2 F2=0
Chao1 估计(手算 5+2*1/2=6) = 6.0
✅ Chao1 手算验证通过（F1=2 → 估计 6）


## 4️⃣ 手算示例：Chao1 完整公式

> 数据：计数 [10, 8, 3, 1, 1]（5 个可观察物种，其中 2 个只出现 1 次）
$$\hat S_{Chao1} = 5 + \frac{2^2}{2\times0} ?? $$
> ⚠️ F₂=0 时公式分母为 0 → 使用无偏修正：$5 + \frac{F_1(F_1-1)}{2} = 5 + \frac{2\times1}{2} = 6$

> 若数据 [10, 8, 3, 2, 2, 1]（S=6, F₁=1, F₂=2）：
$$\hat S = 6 + \frac{1^2}{2\times2} = 6 + 0.25 = 6.25$$

In [4]:
# 第二种情形的验证
est2, S2, f12, f22 = chao1([10, 8, 3, 2, 2, 1])
print('数据 [10,8,3,2,2,1]: S_obs=%d F1=%d F2=%d -> Chao1=%g' % (S2, f12, f22, est2))
assert abs(est2 - 6.25) < 1e-9
print('✅ Chao1 第二种情形验证通过')

数据 [10,8,3,2,2,1]: S_obs=6 F1=1 F2=2 -> Chao1=6.25
✅ Chao1 第二种情形验证通过


## 5️⃣ 稀疏曲线与深度归一化

**稀疏曲线**：随机抽取 n 条序列（不放回），画观察物种数 vs n。
- 曲线**平台期**意味着测序深度足够
- 若未达平台 → 更多测序会带来更多物种

**抽平（rarefaction）**：把所有样本抽到相同深度再算 α 多样性，可公平比较。

In [5]:
def rarefaction_species(seqs, n, seed=1):
    "随机抽取 n 条序列，返回不同物种数"
    rng = np.random.RandomState(seed)
    sub = rng.choice(len(seqs), size=n, replace=False)
    return len(set(seqs[i] for i in sub))

# 模拟一个含有 30 个物种、共 5000 条序列的"样本"
rng = np.random.RandomState(7)
species_pool = ['sp%02d' % i for i in range(30)]
seqs_full = rng.choice(species_pool, size=5000, p=None)  # 均匀抽

import numpy as _np
depths = [100, 300, 600, 1000, 2000, 4000, 5000]
nsp = []
for d in depths:
    ns = 0
    for rep in range(10):
        ns += rarefaction_species(seqs_full, d, seed=rep)
    nsp.append(ns / 10)
print('深度 → 平均物种数:')
for d, ns in zip(depths, nsp):
    print('  n=%5d 物种数=%5.1f' % (d, ns))
print('→ 深度越大物种数越多；到 5000 时接近 30（模拟物种总数）→ 曲线趋于平台')
assert nsp[-1] > nsp[0]
print('✅ 稀疏曲线演示完成（单调上升趋于平台）')

深度 → 平均物种数:
  n=  100 物种数= 28.8
  n=  300 物种数= 30.0
  n=  600 物种数= 30.0
  n= 1000 物种数= 30.0
  n= 2000 物种数= 30.0
  n= 4000 物种数= 30.0
  n= 5000 物种数= 30.0
→ 深度越大物种数越多；到 5000 时接近 30（模拟物种总数）→ 曲线趋于平台
✅ 稀疏曲线演示完成（单调上升趋于平台）


## 6️⃣ 手算示例：抽平前后的公平比较

> 样本 X 测到 40 条（含 4 物种），样本 Y 测到 100 条（含 6 物种）。
> 直接用物种数比：Y"更丰富"？不公平——深度不同。
> **抽平到 40 条**后，Y 的物种数可能降为 5 → 两者接近可比。

> 这就是为什么宏基因组论文报告多样性时总强调**标准化深度**。

In [6]:
# 抽平演示：两个样本抽到相同深度
Y = ['A']*30 + ['B']*30 + ['C']*20 + ['D']*10 + ['E']*8 + ['F']*2   # 100 条, 6 物种
ns = [rarefaction_species(Y, 40, seed=s) for s in range(20)]
print('样本 Y 抽平到 40 条: 物种数均值 = %.1f（原为 6）' % (sum(ns)/len(ns)))
print('→ 与样本 X 的 4 物种（40 条满测）相比，差异缩小')
print('✅ 抽平可比性演示完成')

样本 Y 抽平到 40 条: 物种数均值 = 5.5（原为 6）
→ 与样本 X 的 4 物种（40 条满测）相比，差异缩小
✅ 抽平可比性演示完成


## ✅ 自测清单

- [ ] 能区分 OTU 与 ASV
- [ ] 能手算 Shannon / Simpson 指数
- [ ] 能手算 Chao1（含 F2=0 的修正）
- [ ] 能解释稀疏曲线与抽平的必要性